# 01 — Finite Field Arithmetic

Everything in Nova happens over a **finite field** $\mathbb{F}_p$ — a set of integers $\{0, 1, \ldots, p-1\}$ where all arithmetic is done modulo a prime $p$.

**Why finite fields?** Cryptographic proof systems need:
- Exact arithmetic (no floating-point errors)
- Random sampling from a known set
- The Schwartz-Zippel lemma: two different degree-$d$ polynomials agree on at most $d/p$ fraction of points — so random evaluation catches cheating with overwhelming probability

**Physics analogy:** Think of $\mathbb{F}_p$ as a discrete circular space (like angles mod $2\pi$, but discrete). Addition wraps around, multiplication distributes, and every nonzero element has an inverse.

In [ ]:
import numpy as np
import galois

# We use a 61-bit Mersenne prime: p = 2^61 - 1
# Large enough for security, small enough to fit in a 64-bit int
p = 2**61 - 1
print(f"Our prime: p = 2^61 - 1 = {p}")
print(f"Bits: {p.bit_length()}")

In [ ]:
# Create the Galois field — all arithmetic happens here
GF = galois.GF(p)

# Basic arithmetic: addition wraps mod p
a = GF(p - 1)  # the largest element
b = GF(2)
print(f"(p-1) + 2 = {a + b}")  # wraps to 1
print(f"Expected: {(p - 1 + 2) % p}")

In [ ]:
# Multiplication also wraps
x = GF(1000000)
y = GF(999999)
print(f"{int(x)} * {int(y)} = {x * y}")
print(f"Verification: {(1000000 * 999999) % p}")

In [ ]:
# Every nonzero element has a multiplicative inverse
# a * a^(-1) = 1
a = GF(42)
a_inv = GF(1) / a  # or equivalently: a ** (p - 2) by Fermat's little theorem
print(f"42^(-1) mod p = {a_inv}")
print(f"42 * 42^(-1) = {a * a_inv}")  # should be 1

## Vector Operations

In Nova, we work with vectors and matrices over $\mathbb{F}_p$. The key operations are:
- **Hadamard (element-wise) product**: $\mathbf{a} \circ \mathbf{b}$ — multiply corresponding elements
- **Inner product**: $\langle \mathbf{a}, \mathbf{b} \rangle = \sum a_i b_i$
- **Matrix-vector product**: $A\mathbf{z}$ — the usual matrix multiplication, but over $\mathbb{F}_p$

In [ ]:
# Vectors over GF(p)
v1 = GF(np.array([1, 2, 3, 4]))
v2 = GF(np.array([5, 6, 7, 8]))

# Hadamard product (element-wise) — this is the ∘ in R1CS
hadamard = v1 * v2
print(f"v1 ∘ v2 = {hadamard}")  # [5, 12, 21, 32]

# Inner product
inner = np.sum(v1 * v2)
print(f"<v1, v2> = {inner}")  # 5 + 12 + 21 + 32 = 70

In [ ]:
# Matrix-vector product over GF(p)
M = GF(np.array([[1, 0, 1],
                  [0, 1, 1]]))
v = GF(np.array([3, 5, 7]))

result = M @ v
print(f"M @ v = {result}")  # [3+7, 5+7] = [10, 12]

## Random Sampling

Zero-knowledge proofs rely heavily on random challenges. The Fiat-Shamir heuristic derives these from a hash, but the underlying property is that a random field element is uniformly distributed over $\{0, \ldots, p-1\}$.

**Schwartz-Zippel guarantee:** If two degree-$d$ polynomials differ, a random evaluation point catches this with probability $\geq 1 - d/p$. With $p \approx 2^{61}$ and $d$ in the thousands, this is negligibly small.

In [ ]:
from nano_nova.field import random_field_element, random_field_vector

# Sample random elements
r = random_field_element()
print(f"Random element: {r}")

# Sample a random vector
rv = random_field_vector(5)
print(f"Random vector: {rv}")

# The probability of two random elements being equal: 1/p ≈ 2^{-61}
print(f"\nCollision probability: 1/p ≈ 2^(-{p.bit_length()})")
print("This is astronomically small — comparable to SHA-256 collision probability.")

## Key Takeaway

All of Nova's machinery — R1CS constraints, folding, commitments — operates over $\mathbb{F}_p$. The field gives us:
1. **Exact arithmetic** with no rounding errors
2. **Random challenges** that are statistically independent
3. **The Schwartz-Zippel lemma** for soundness guarantees

Next: [02_r1cs_basics.ipynb](02_r1cs_basics.ipynb) — how to encode computations as R1CS constraints.